# DAML Homework 2 Runway — Supervised Learning in 12 Minutes

**Ungraded · no submission required · approximately 10–15 minutes**

This notebook gives you the minimum vocabulary and NumPy moves needed for **Night Chorus**. It is deliberately small: features and labels, train/validation/test splits, accuracy, nearest neighbors, logistic probabilities, and Gini impurity.

<div style="border-left: 5px solid #2f855a; padding: 0.8em 1em; background: #f0fff4;">
<b>The one-line supervised-learning story</b><br><br>
We observe features <code>X</code>, observe labels <code>y</code> during training, learn a rule from them, and use that rule to predict labels for new rows of <code>X</code>.
</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. Features `X` and labels `y`

Each **row** of `X` is one example. Each **column** is one measured feature. The matching entry in `y` is that row's label.

In [ ]:
# Four audio clips, each represented by two acoustic features.
X = np.array([
    [1.0, 1.4],
    [1.2, 1.1],
    [4.1, 4.4],
    [4.5, 4.0],
])
y = np.array([0, 0, 1, 1])  # 0 = insect, 1 = frog

print("X shape:", X.shape, "= (examples, features)")
print("y shape:", y.shape, "= one label per example")
print("First example:", X[0], "label:", y[0])

## 2. Train, validation, and test have different jobs

| Split | Job | Safe question to ask |
|---|---|---|
| **Train** | Learn parameters or store examples | “What rule fits these examples?” |
| **Validation** | Choose settings or choose among models | “Which candidate should I keep?” |
| **Test** | Report one final estimate | “How did the chosen workflow generalize?” |

A useful mnemonic is **train learns, validation chooses, test reports**. Repeatedly changing a model after inspecting test performance quietly turns the test set into another validation set.

## 3. Accuracy is the mean of Boolean comparisons

For classification,

$$
\mathrm{accuracy}=\frac{1}{n}\sum_{i=1}^n \mathbf{1}(\hat y_i=y_i).
$$

In NumPy, `True` behaves like `1` and `False` behaves like `0` when averaged.

In [ ]:
y_true = np.array([0, 1, 1, 0])
y_pred = np.array([0, 1, 0, 0])

correct = (y_true == y_pred)
print("Correct row by row:", correct)
print("Accuracy:", np.mean(correct))

## 4. K-nearest neighbors: distance, sort, vote

KNN does not fit a line. For a new point, it:

1. measures its distance to every training point;
2. sorts those distances;
3. looks at the first `k` labels;
4. takes a majority vote.

In [ ]:
X_train = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 2.0], [3.0, 3.0]])
y_train = np.array([0, 0, 1, 1])
query = np.array([0.9, 0.1])

distances = np.sqrt(np.sum((X_train - query) ** 2, axis=1))
order = np.argsort(distances)
nearest_three = order[:3]

print("Distances:", distances)
print("Indices from nearest to farthest:", order)
print("Three nearest labels:", y_train[nearest_three])
print("Majority-vote prediction:", int(np.mean(y_train[nearest_three]) >= 0.5))

## 5. Logistic regression: score → probability → label

Logistic regression first computes a linear score,

$$
z=Xw+b,
$$

then turns it into a probability with the sigmoid,

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

A threshold such as `0.5` converts the probability into a class label.

In [ ]:
scores = np.array([-2.0, 0.0, 2.0])
probabilities = 1 / (1 + np.exp(-scores))
labels = (probabilities >= 0.5).astype(int)

print("Scores:", scores)
print("Probabilities:", probabilities)
print("Labels at threshold 0.5:", labels)

## 6. Decision trees ask whether a split makes groups purer

For binary labels, Gini impurity is

$$
G=1-p_0^2-p_1^2.
$$

A leaf containing one class has impurity `0`. A 50/50 binary leaf has impurity `0.5`. Trees search for splits with low weighted child impurity.

In [ ]:
def demo_gini(labels):
    labels = np.asarray(labels)
    p1 = np.mean(labels == 1)
    p0 = 1 - p1
    return 1 - p0**2 - p1**2

print("Pure leaf [1, 1, 1]:", demo_gini([1, 1, 1]))
print("Balanced leaf [0, 0, 1, 1]:", demo_gini([0, 0, 1, 1]))

## 7. Two tiny Python patterns used in the homework

In [ ]:
# Pattern A: select the smallest k tied for the best validation score.
k_values = np.array([1, 3, 5, 7])
scores = np.array([0.80, 0.90, 0.90, 0.85])
best_score = np.max(scores)
tied_ks = k_values[scores == best_score]
best_k = int(np.min(tied_ks))
print("Best k:", best_k)

# Pattern B: choose a model by its validation score.
model_scores = {"KNN": 0.91, "Logistic": 0.78, "Tree": 0.86}
chosen = max(model_scores, key=model_scores.get)
print("Chosen model:", chosen)

## Ready check

You are ready for the homework if these statements feel reasonable:

- `X.shape == (n_examples, n_features)`;
- `y.shape == (n_examples,)`;
- validation data chooses settings; test data reports the final result;
- KNN uses nearby labels, logistic regression uses a probability from a linear score, and a tree uses threshold rules;
- lower Gini impurity means a purer node.

Keep this notebook open beside the assignment if any of those ideas still feel slippery.